# Advanced Residual Stream Decomposition

Deep analysis of residual stream structure: orthogonal decomposition, per-token buildup tracking, component interference patterns, subspace evolution, and contribution isolation.

## Why This Matters

Composition residual decomposition advanced measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_decomposition_advanced import (
    orthogonal_decomposition,
    per_token_residual_buildup,
    component_interference,
    residual_subspace_tracking,
    contribution_isolation,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

In [ ]:
# Orthogonal decomposition of the residual stream
result = orthogonal_decomposition(model, tokens)
print(f'Components: {len(result["components"])}')
print(f'Explained variance: {result["explained_variance"]:.4f}')
print(f'Residual norm: {result["residual_norm"]:.4f}')
for comp in result['components'][:3]:
    print(f'  {comp["name"]}: projection={comp["projection"]:.4f}')

In [ ]:
# Track how the residual stream builds up token by token
result = per_token_residual_buildup(model, tokens)
print('Residual norms per layer:', np.array(result['residual_norms']).round(3))
print('Attn contributions:', np.array(result['attn_contributions']).round(3))
print('MLP contributions:', np.array(result['mlp_contributions']).round(3))
print('Cumulative buildup:', np.array(result['cumulative_buildup']).round(3))

In [ ]:
# Measure interference between components
result = component_interference(model, tokens)
print(f'Interference matrix shape: {result["interference_matrix"].shape}')
print(f'Net interference: {result["net_interference"]:.4f}')
print(f'Constructive pairs: {len(result["constructive_pairs"])}')
print(f'Destructive pairs: {len(result["destructive_pairs"])}')
print(f'Labels: {result["labels"][:4]}')

In [ ]:
# Track subspace evolution across layers
result = residual_subspace_tracking(model, tokens)
print(f'Subspace overlap matrix shape: {result["subspace_overlap"].shape}')
print(f'Effective rank per layer: {np.array(result["effective_rank"]).round(2)}')
print('Overlap matrix (first 3x3):')
print(np.array(result['subspace_overlap'][:3, :3]).round(3))

In [ ]:
# Isolate a specific component's contribution
result = contribution_isolation(model, tokens, 'blocks.0.hook_attn_out')
print(f'Contribution norm: {result["contribution_norm"]:.4f}')
print(f'Top promoted tokens: {result["promoted_tokens"][:3]}')
print(f'Top demoted tokens: {result["demoted_tokens"][:3]}')